# Seismic events overlaid on a mine map

In [ ]:
import matplotlib.pyplot as plt

import geopandas as gpd
import numpy as np

In [ ]:
thickness_between = gpd.read_file('mocnost mezilozi lines.shp')

# thickness_between.plot()
# plt.title('Mocnost meziloží')
# plt.show()

In [ ]:
from seismi.dxf import load_dxf

mines_map = load_dxf('mines_map.dxf')

In [ ]:
from seismi.xls import read_seismic_events, read_measured_points, read_interpolated

seismic_events = read_seismic_events('SL jevy pri dobyvaní R 140 704.xlsx')
depths = read_measured_points('sloj 40 hloubka vrty.xlsx')
depths_interpolated = read_interpolated('sloj 40 hloubka grid.xlsx')

In [ ]:
from seismi.plots import plot_basic_map
fig, ax = plot_basic_map(mines_map, seismic_events)
plt.show()

Now this will be out of alignment:

In [ ]:
fig, ax = plot_basic_map(mines_map, seismic_events, thickness=thickness_between)
plt.show()

In [ ]:
from seismi.plots import create_interactive_map

m = create_interactive_map(
    seismic_events,
    depths=depths,
    depths_interpolated=depths_interpolated,
    sample_events=200,
    sample_interpolated=1000,
)
m

In [ ]:
from seismi.nn import prepare_data, BiggerNN, train_model, evaluate_model

(X_train, X_test, y_train, y_test,
 X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
 scaler) = prepare_data(seismic_events, depths_interpolated)

In [ ]:
from seismi.nn import BiggerNN, IndexedDataset, SpatialMSELoss, train_model, evaluate_model
import torch

# Get original (unscaled) coordinates for spatial calculations
X_train_orig = scaler.inverse_transform(X_train)
X_test_orig = scaler.inverse_transform(X_test)

# Prepare DataLoader with indices for spatial loss
dataset = IndexedDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

# Create spatial loss criterion
criterion = SpatialMSELoss(
    coords=X_train_orig,
    targets=y_train,
    n_neighbors=5,
    alpha=2
)

# Train model with spatial loss
spatial_model = BiggerNN()
spatial_losses = train_model(
    spatial_model,
    data_loader,
    criterion,
    epochs=5,
)

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(spatial_losses)
plt.xlabel('Epoch')
plt.ylabel('Spatial Loss')
plt.title('Training Loss (Spatial)')
plt.grid(True)
plt.show()

# Evaluate with standard and spatial metrics
spatial_results = evaluate_model(
    spatial_model,
    torch.tensor(X_test, dtype=torch.float32),
    y_test,
    coords=X_test_orig,
    log_transform=True,
    spatial_metrics=True,
    n_neighbors=5
)

spatial_preds = spatial_results['predictions']

# Visualize results
plt.figure(figsize=(12, 10))

# 1. True vs Predicted
plt.subplot(221)
plt.scatter(y_test, spatial_preds, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Spatial Model Predictions')
plt.grid(True)

# 2. Spatial error distribution
plt.subplot(222)
plt.scatter(
    X_test_orig[:, 0],
    X_test_orig[:, 1],
    c=spatial_results['local_error_pattern'],
    cmap='coolwarm',
    s=50,
    alpha=0.7
)
plt.colorbar(label='Error Compared to Neighbors')
plt.title('Spatial Error Distribution')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.grid(True)

# 3. Histogram of spatial errors
plt.subplot(223)
spatial_errors = np.abs(spatial_preds - y_test)
plt.hist(spatial_errors, alpha=0.7, bins=20)
plt.xlabel('Absolute Error')
plt.ylabel('Frequency')
plt.title('Spatial Model Error Distribution')
plt.grid(True)

# 4. Neighborhood visualization for a random point
plt.subplot(224)
random_point = np.random.randint(0, len(X_test_orig))
neighbors = spatial_results['neighbor_indices'][random_point]
neighbor_coords = X_test_orig[neighbors]
point_coords = X_test_orig[random_point]

plt.scatter(X_test_orig[:, 0], X_test_orig[:, 1], c='gray', alpha=0.2, s=10)
plt.scatter(neighbor_coords[:, 0], neighbor_coords[:, 1], c='blue', s=50, label='Neighbors')
plt.scatter(point_coords[0], point_coords[1], c='red', s=100, label='Selected Point')
for neighbor in neighbor_coords:
    plt.plot([point_coords[0], neighbor[0]], [point_coords[1], neighbor[1]], 'k--', alpha=0.4)
plt.title('Neighborhood Connections')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create interactive map with model predictions
from seismi.plots import create_interactive_prediction_map

spatial_map = create_interactive_prediction_map(
    X_test, y_test, spatial_preds, scaler,
)
spatial_map